# Preprocessing

In [1]:
from Load import OSMBuildingReader

In [2]:
cada_reader = OSMBuildingReader(encoding="utf-8")
cada_path = "./data/Cadastre_buildings.shp"
cada_reader.read_osm_buildings(cada_path)
cada_reader.print_info()

INFO:Load.shp_reader:正在读取shapefile: data/Cadastre_buildings.shp
INFO:Load.shp_reader:成功读取 4905 个要素
INFO:Load.shp_reader:原始坐标系: EPSG:32618
INFO:Load.shp_reader:转换坐标系: EPSG:32618 -> EPSG:3857
INFO:Load.shp_reader:坐标系转换完成
INFO:Load.shp_reader:移除无效几何对象后，剩余 4879 个要素


Shapefile info
文件路径: data/Cadastre_buildings.shp
要素数量: 4905
列数: 3
列名: ['BuildingId', 'height', 'geometry']
原始坐标系: EPSG:32618
当前坐标系: EPSG:3857
边界框: [-8416068.889547948, 695887.7617808486, -8413363.225300174, 697174.1225201517]
几何类型: {'Polygon': 4561, 'MultiPolygon': 344}
空几何数量: 0
无效几何数量: 26


In [3]:
# Load the mask shapefile to filter the range
group_reader = OSMBuildingReader(encoding="utf-8")
group_path = "./data/study_area.shp"
group_reader.read_osm_buildings(group_path)
group_gdf = group_reader.data
area_mask = group_reader.data[group_reader.data['id'] == 2].geometry.iloc[0]
print(type(area_mask))

INFO:Load.shp_reader:正在读取shapefile: data/study_area.shp
INFO:Load.shp_reader:成功读取 3 个要素
INFO:Load.shp_reader:原始坐标系: EPSG:3857
INFO:Load.shp_reader:移除无效几何对象后，剩余 3 个要素


<class 'shapely.geometry.polygon.Polygon'>


## Filter within the mask

In [4]:
cada_reader.filter_by_polygon(area_mask)
cada_reader.print_info()

INFO:Load.shp_reader:Polygon filter: 854 -> 854 items


Shapefile info
文件路径: data/Cadastre_buildings.shp
要素数量: 854
列数: 3
列名: ['BuildingId', 'height', 'geometry']
原始坐标系: EPSG:32618
当前坐标系: EPSG:3857
边界框: [-8414554.955579014, 695943.7136334346, -8413363.225300174, 696758.7128561819]
几何类型: {'Polygon': 812, 'MultiPolygon': 42}
空几何数量: 0
无效几何数量: 8


## Clean and split

In [5]:
cada_reader.split_multipolygons()
cada_reader.add_centroid_column()
gdf = cada_reader.clean_data
cada_reader.calculate_node_statistics()
gdf

INFO:Load.shp_reader:移除无效几何对象后，剩余 846 个要素
INFO:Load.shp_reader:分割multipolygons后，共有 894 个要素
INFO:Load.shp_reader:已添加质心列
INFO:Load.shp_reader:Num of nodes =25522, Average node=28.55


Num of nodes: 25522
Num of buildings: 894
Average node per building: 28.55
Least num of node: 4
Most num of node: 677
Med num of node: 22.0


,BuildingId,height,geometry,ori_id,centroid
3,110500300270000,14.0,"POLYGON ((-8413918.398 695996.547, -8413918.30...",110500300270000,POINT (-8413920.912 695993.526)
4,110505200240000,20.0,"POLYGON ((-8413990.066 696473.458, -8413993.32...",110505200240000,POINT (-8413993.085 696485.465)
4,110505200240000,20.0,"POLYGON ((-8413956.05 696509.402, -8413952.771...",110505200240000,POINT (-8413960.005 696507.724)
10,110505900150000,20.0,"POLYGON ((-8413901.426 696570.167, -8413902.35...",110505900150000,POINT (-8413897.796 696576.849)
13,110502200010000,20.0,"POLYGON ((-8413939.823 696080.642, -8413939.78...",110502200010000,POINT (-8413943.431 696090.969)
...,...,...,...,...,...
4883,110502300160000,14.0,"POLYGON ((-8414042.442 696095.094, -8414045.52...",110502300160000,POINT (-8414046.622 696098.871)
4885,110506000120000,11.0,"POLYGON ((-8413844.801 696589.32, -8413845.457...",110506000120000,POINT (-8413840.542 696597.721)
4887,110502800030000,11.0,"POLYGON ((-8414166.875 696264.527, -8414167.22...",110502800030000,POINT (-8414163.269 696277.051)
4902,110500400080000,12.0,"POLYGON ((-8414151.762 695957.224, -8414154.78...",110500400080000,POINT (-8414153.398 695962.394)


## Fill the hole and then Simplify the polygons

In [6]:
cada_reader.fill_holes_and_simplify()
cada_reader.calculate_node_statistics()
gdf = cada_reader.clean_data
gdf

INFO:Load.shp_reader:Fill holes and simplify: 894 -> 894 items
INFO:Load.shp_reader:Num of nodes =11089, Average node=12.4


Num of nodes: 11089
Num of buildings: 894
Average node per building: 12.4
Least num of node: 4
Most num of node: 68
Med num of node: 11.0


,BuildingId,height,geometry,ori_id,centroid
3,110500300270000,14.0,"POLYGON ((-8413918.398 695996.547, -8413918.16...",110500300270000,POINT (-8413920.912 695993.526)
4,110505200240000,20.0,"POLYGON ((-8413990.066 696473.458, -8413993.32...",110505200240000,POINT (-8413993.085 696485.465)
4,110505200240000,20.0,"POLYGON ((-8413956.05 696509.402, -8413952.771...",110505200240000,POINT (-8413960.005 696507.724)
10,110505900150000,20.0,"POLYGON ((-8413901.426 696570.167, -8413902.35...",110505900150000,POINT (-8413897.796 696576.849)
13,110502200010000,20.0,"POLYGON ((-8413939.823 696080.642, -8413939.78...",110502200010000,POINT (-8413943.431 696090.969)
...,...,...,...,...,...
4883,110502300160000,14.0,"POLYGON ((-8414042.442 696095.094, -8414048.89...",110502300160000,POINT (-8414046.622 696098.871)
4885,110506000120000,11.0,"POLYGON ((-8413844.801 696589.32, -8413845.457...",110506000120000,POINT (-8413840.542 696597.721)
4887,110502800030000,11.0,"POLYGON ((-8414166.875 696264.527, -8414167.22...",110502800030000,POINT (-8414163.269 696277.051)
4902,110500400080000,12.0,"POLYGON ((-8414151.762 695957.224, -8414154.78...",110500400080000,POINT (-8414153.398 695962.394)


## Save result to a new shapefile

In [7]:
cada_reader.export_clean_to_file("./data/id3_cleaned_simplified.shp")

INFO:pyogrio._io:Created 894 records
INFO:Load.shp_reader:数据已导出到: ./data/id3_cleaned_simplified.shp
